# Graph Neural Networks

Welcome to the third laboratory class of ML4HD course!

**Today you will see:**
- an example of audio data pre-processing
- an implementation of a Graph Neural Network.

**After this assignment you will be able to:**

- load and transform the data using the TensorFlow input pipeline
- build and train a GNN in TensorFlow for a multi-class classification problem (keyword spotting)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd '/content/drive/MyDrive/MLHD_labs/Lab_4'

/content/drive/MyDrive/MLHD_labs/Lab_4


Decomment the following lines to install the libraries.

In [ ]:
%%capture

!pip install tensorflow_io
!pip install tensorflow_gnn
!pip install tf_keras

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # needed for tfgnn
import time
from tqdm import tqdm
import pandas as pd
import tensorflow as tf
import tensorflow_io as tfio
import tensorflow_gnn as tfgnn
import random
import numpy as np
import time
from pathlib import Path
import matplotlib.pyplot as plt

from utils_lab import *
from plotting_utils import plot_recording_waveform, plot_spectrogram_and_mfccs, plot_prediction_distribution

import matplotlib as mpl
mpl.rcParams['figure.figsize'] = (10, 6)
mpl.rcParams['axes.grid'] = True
mpl.rcParams['legend.fontsize'] = 'large'

/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl5mutex6unlockEv']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZN3tsl7strings13safe_strtou64ESt17basic_string_viewIcSt11char_traitsIcEEPm']
  warnings.warn(

In [ ]:
seed = 42

# setting seed for reproducibility of random operations
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

 ## 1 - Data loading and preprocessing
 The **Speech Commands Dataset** v0.02 includes one-second audio recordings of spoken English words designed for training and evaluating _keyword-spotting_ models. The words cover a fixed vocabulary of simple commands such as _yes_, _no_, _down_, _left_, _right_, and _stop_, as well as background noise and ten auxiliary unknown-word examples.

 More information about the dataset can be found here: [Speech Commands: A Dataset for Limited-Vocabulary Speech Recognition](https://arxiv.org/abs/1804.03209)

 ### 1.1 - Create data folder
 First, if not done yet, let's decompress the zip data file.

In [ ]:
## Unzip the data from the shared Datasets folder into local "/content/Data" folder.
!mkdir -p '/content/Data'
!tar -xzf "/content/drive/MyDrive/MLHD_labs/Datasets/Lab_4/speech_commands_v0.02.tar.gz" -C "/content/Data" --skip-old-files

### 1.2 - Load the data using the official split provided by text files

We use the provided text files to define the test and validation splits: any file listed in *testing_list.txt* goes into the test set, and any file in *validation_list.txt* goes into the validation set. All remaining audio files are automatically assigned to the training set.

After splitting the data, we build a *class-to-index* mapping by assigning each class folder a unique integer label, which allows us to convert folder names into numerical labels suitable for model training.

In [ ]:
# create useful variables to store paths to folders and txt files
data_dir = Path('/content/Data')
validation_samples = 'validation_list.txt'
test_samples = 'testing_list.txt'
background_noise_dir = os.path.join(data_dir, '_background_noise_')
background_noise_files = tf.io.gfile.glob(str(Path(background_noise_dir) / '*.wav'))

To load the data we will use the already implemented function  `load_audio_dataset()`, implemented in the  `utils_lab.py` file.

In [ ]:
# Load data
train_files, train_labels, train_labels_str, val_files, val_labels, val_labels_str, test_files, test_labels, test_labels_str, classes = load_audio_dataset(
            data_dir=data_dir,
            validation_samples=os.path.join(data_dir, validation_samples),
            test_samples=os.path.join(data_dir, test_samples)
        )

In [ ]:
print(train_files[:5])

In [ ]:
# print dataset statistics
total = len(train_files) + len(val_files) + len(test_files)
print(f"Percentage of train samples: {len(train_files)/total*100:.1f}%")
print(f"Percentage of validation samples: {len(val_files)/total*100:.1f}%")
print(f"Percentage of test samples: {len(test_files)/total*100:.1f}%")
print("\nTotal Number Samples :" , total)

### 1.3 Builing a reference Dataframe

To keep our dataset organized, we want a single dataframe (i.e., table) containing:

- the **file path** of each audio file
- its **label index**
- the **data split** it belongs to (i.e., *train*, *val*, or *test*)

We can do this using `pandas.DataFrame()`, which allows us to create structured data tables.

**For each split** (train/val/test), we create a DataFrame like this:

```
pd.DataFrame({
    'file_path': ...,   # list of file paths
    'label': ...,       # list of numerical labels
    'label_str': ...,   # list of string labels
    'split':            # 'train'/'val'/'test'
})

```
We then use `pd.concat()` to combine the three dataframes into a single one.

In [ ]:
# Create a dataframe containing all the file paths, the labels from the reference file
# START CODE HERE ### (3 lines)
train_df = None

val_df = None

test_df = None
### END CODE HERE ###

# Concatenate all splits into a single reference dataframe
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

In [ ]:
df.head()

In [ ]:
# plot the labels distribution per split
df.groupby(["split", "label_str"]).size().unstack("split").plot(kind="bar", figsize=(10, 3))
plt.ylabel("Count")
plt.title("Label distribution per split")
plt.show()

### 1.4 - Preprocessing on audio signals
To ensure consistency and improve model robustness, each audio sample undergoes a series of preprocessing steps:

1. **Read and decode** the wav file implementing the `read_path_to_wav()` function. Specifically, use `tf.io.read_file(file_path)` to read the file path, and then use `tf.audio.decode_wav(file_contents)` to decode tha .wav file. Be aware that `tf.audio.decode_wav()` **returns two elements**: the audio and the sample rate.
2. **Adjust the audio length** to 1 second (corresponding to 16,000 samples at 16 kHz) using the `adjust_audio_length()` function. We will **trim** the audio if **it is too long** by taking the first 16,000 samples, and we will **pad** it if **it is too short** by adding the correct amount of zeros at the end of the audio using the function `tf.pad()`.
3. Apply a random time shift with the `apply_time_shift()` function using the `tf.roll()` function. Remember to specify the correct axis inside the tf.roll() function.
4. **Add random noise** with the `apply_random_noise()` function using samples taken from the _ _background_noise__ folder.

In [ ]:
# Define global variables
SAMPLE_RATE = 16000 # 16 KHz
FRAME_LENGTH = int(SAMPLE_RATE * 0.025)  # 25 ms
FRAME_STEP = int(SAMPLE_RATE * 0.010)  # 10 ms

In [ ]:
def read_path_to_wav(file_path):
    # START CODE HERE ### (2 lines)
    file_contents = None
    wav, _ = None
    ### END CODE HERE ###
    return wav

def adjust_audio_length(wav, sample_rate):
  target_length = sample_rate
  # START CODE HERE ### (1 line)
  current_length = None
  ### END CODE HERE ###
  if current_length > target_length:
      # START CODE HERE ### (1 line)
      wav = None
      ### END CODE HERE ###
  else:
      # START CODE HERE ### (2 lines)
      paddings = [[0, None], [0, 0]] # substitute None with the correct amount of zeros
      wav = None
      ### END CODE HERE ###

  return tf.squeeze(wav, axis=-1)

def apply_time_shift(wav, sample_rate, max_time_shift_ms = 100):
    max_shift_samples = int((max_time_shift_ms / 1000.0) * sample_rate)
    shift_samples = tf.random.uniform(shape=[], minval=-max_shift_samples, maxval=max_shift_samples + 1, dtype=tf.int32)
    # START CODE HERE ### (1 line)
    shifted_audio = None
    ### END CODE HERE ###
    return shifted_audio

In [ ]:
def apply_random_noise(wav, background_noise_files, target_length, noise_prob, min_snr_db=0, max_snr_db=10):

    background_noise_files = tf.convert_to_tensor(background_noise_files, dtype=tf.string)

    # Select a random noise file
    noise_file = tf.random.shuffle(background_noise_files)[0]
    noise_wav = read_path_to_wav(noise_file)

    # Extract a random segment of the desired length
    noise_len = tf.shape(noise_wav)[0]
    max_start = tf.maximum(noise_len - target_length, 1)
    start = tf.random.uniform([], 0, max_start, dtype=tf.int32)
    noise = noise_wav[start : start + target_length]

    noise = tf.squeeze(noise, axis=-1) if tf.rank(noise) > 1 else noise

    # Compute scaling for SNR
    signal_power = tf.reduce_mean(tf.square(wav))
    noise_power = tf.reduce_mean(tf.square(noise))

    snr_db = tf.random.uniform([], min_snr_db, max_snr_db)
    snr_lin = tf.pow(10.0, snr_db / 10.0)

    scale = tf.sqrt(signal_power / (noise_power * snr_lin + 1e-12))
    noise = noise * scale

    # Apply noise with given probability
    do_apply = tf.random.uniform([]) <= noise_prob
    noise_signal = tf.where(do_apply, wav + noise, wav)
    return noise_signal

Let's plot the waveform of a sample recording along with the waveform of the same sample after shifting and noise injection.

In [ ]:
train_sample = train_df['file_path'].sort_values()[0]
train_sample

In [ ]:
audio_sample = adjust_audio_length(read_path_to_wav(train_sample), SAMPLE_RATE)
shifted_audio_sample = apply_time_shift(audio_sample, SAMPLE_RATE)
noisy_audio_sample = apply_random_noise(shifted_audio_sample, background_noise_files, SAMPLE_RATE, 1)

plot_recording_waveform(audio_sample ,noisy_audio_sample, SAMPLE_RATE)

### 1.5 - Preprocessing: convert waveforms to spectrograms

The waveforms in the dataset are represented in the time domain. Next, you'll transform the waveforms from the time-domain signals into the time-frequency-domain signals by computing the **Short-Time Fourier Transform (STFT)** to convert the audio into spectrograms, which show frequency changes over time and can be represented as 2D images.

The STFT splits the audio signal $x[n]$ into overlapping windows of time (or *frames*) and applies the Fourier transform on each frame.

We will use the `tf.signal.stft` function provided by TensorFlow. This function takes as arguments: the `signal`, the `frame_length`, the `frame_step`, the `fft_length` and the `window_fn`.

- The `frame_length` is the window length in samples. In our case, we will use a frame length of 25 ms (wich translates into 400 samples at 16 kHz);
- The `frame_step` is the number of samples to step. (which determines the size and overlap of the STFT windows). In our case, we will use a step of 10 ms (160 samples at 16 kHz).
- The `fft_length` is the size of the FFT to apply. We will **not** specify this value, and TensorFlow, by default, will use the smallest power of 2 enclosing frame_length.
- The `window_fn` is the windowing function use, in our case: `tf.signal.hamming_window`.

**Note on the window**: When performing a STFT, we analyze the signal in small overlapping frames. If we apply the FFT directly to each frame (using a rectangular window), we would implicitly assume the signal is **periodic within that frame** — which is almost never true.
This causes spectral leakage, i.e., frequency energy from one component “leaks” into neighboring frequency bins, making peaks blurrier and less accurate. The Hamming window smooths the beginning and end of each frame by down-weighting samples near the edges, by reducing discontinuity at frame boundaries.

Finally, the STFT produces an array of **complex numbers** representing magnitude and phase. In this lab, we are interested only in its *magnitude*, which we can obtain by applying `tf.abs`.


In [ ]:
def get_spectrogram(wav, sample_rate):

    frame_length = int(sample_rate * 0.025)
    frame_step = int(sample_rate * 0.010)

    # START CODE HERE ### (1 line)
    spectrogram = None
    ### END CODE HERE ###

    # Obtain the magnitude of the STFT.
    # START CODE HERE ###
    spectrogram = None
    ### END CODE HERE ###

    return spectrogram, wav

### 1.6 - Preprocessing: compute Mel filters, MFCC, and delta features

**MFCCs (Mel-Frequency Cepstral Coefficients)** represent the audio by mimicking how the human ear perceives sound.
We start by computing the **log-mel spectrogram**, which gives information abot the amount of energy at each frequency band. A **Discrete Cosine Transform (DCT)** is then applied, yielding the initial base coefficients ($\text{mfccs}_0$) that describe the sound's pitch. To capture temporal changes, the **first derivative (delta, $\text{mfccs}_{\text{delta}\_1}$)** and **second derivative (delta-delta, $\text{mfccs}_{\text{delta}\_2}$)** are computed. The final MFCC feature vector is the **concatenation** of these three components.

In the following we will implement the following functions:
- `apply_mel_filterbanks()`, where you will create the Mel filterbanks by using the `tf.signal.linear_to_mel_weight_matrix()`. This function takes as input: the number of Mel filters, the number of spectrogram bins, the sample rate, the minimu and the maximum frequency. You will then apply the tranformation using `tf.tensordot(spectrogram, mel_filterbank, 1)`.
- `compute_delta()` to compute the delta coefficients.
- `get_mfccs()` to obtain the final MFCCs.
  - You will use the tf.signal.`mfccs_from_log_mel_spectrograms()` function applied on the Log Mel spectrogram to obtain the DCT coefficients, from which you have to **discard the first one**.
  - Then you will compute the first and second derivative of the MFCCs using the `compute_delta()` function.
  - Then you will compute the log energy of each signal frame.
  - You will compute the first and second derivative of the energy signal using the `compute_delta()` function.
  - Finally, you will concatenate the coefficients to obtain the final MFCCs.



In [ ]:
def apply_mel_filterbanks(spectrogram, wav, sample_rate):

    # Obtain the number of frequency bins of our spectrogram.
    num_spectrogram_bins = tf.shape(spectrogram)[-1]

    # Define the frequency band we are intereted in
    min_frequency = 100
    max_frequency = sample_rate/2

    # And the number of filters
    num_mel_filters = 13

    # Create Mel filterbank
    # START CODE HERE ### (1 line)
    mel_filterbank = None
    ### END CODE HERE ###

    # Apply the transformation
    # START CODE HERE ### (1 line)
    mel_spectrogram = None
    ### END CODE HERE ###

    # Set output shape
    output_shape = tf.concat([tf.shape(spectrogram)[:-1], [tf.shape(mel_filterbank)[-1]]], axis=0)
    mel_spectrogram = tf.reshape(mel_spectrogram, output_shape)

    # Compute a stabilized log to get log-magnitude mel-scale spectrogram
    log_mel_spectrogram = tf.math.log(mel_spectrogram + np.finfo(float).eps)


    return log_mel_spectrogram, wav


In [ ]:
def compute_delta(mfccs, M):

        frame_count = tf.shape(mfccs)[0]
        # Pad the mfccs at the beginning and at the end to handle boundary frames
        padded_mfccs = tf.pad(mfccs, [[M, M], [0, 0]], mode='SYMMETRIC')    # This pads [M,M] in time (frames) dimension and pads [0,0] in the frequency dimension

        denominator = 2 * sum([m**2 for m in range(1, M+1)])
        # Initialize the deltas
        deltas = tf.zeros_like(mfccs)

        for m in range(1, M+1):
            # Get frames at n+m
            next_frames = padded_mfccs[M + m: M + m + frame_count]
            # Get frames at n-m
            prev_frames = padded_mfccs[M - m : M - m + frame_count]
            # Add weighted difference to the delta coefficients
            deltas += m * (next_frames - prev_frames) / denominator

        return deltas

def get_mfccs(log_mel_spectrogram, wav, frame_length=FRAME_LENGTH, frame_step=FRAME_STEP, M = 2):

    # compute DCT and select coefficients, discarding the first
    # START CODE HERE (1 line)
    mfccs_0 = None
    ### END CODE HERE

    # compute the first derivative of the MFCCs
    # START CODE HERE (1 line)
    mfccs_delta_1 = None
    ### END CODE HERE

    # compute the second derivative of the MFCCs
    # START CODE HERE (1 line)
    mfccs_delta_2 = None
    ### END CODE HERE

    # Divide the raw audio into frames
    framed_wav = tf.signal.frame(wav, frame_length= frame_length, frame_step= frame_step)

    # Compute the log energy of each frame
    frame_energy = tf.reduce_sum(framed_wav**2, axis=-1)
    log_frame_energy = tf.math.log(frame_energy + np.finfo(float).eps)/tf.math.log(10.0)
    log_frame_energy = tf.expand_dims(log_frame_energy, axis=-1)

    # Compute second energy
    # START CODE HERE (1 line)
    energy_delta_1 = None
    ### END CODE HERE

    # Compute third energy
    # START CODE HERE (1 line)
    energy_delta_2 = None
    ### END CODE HERE

    # obtain final MFCCs
    mfccs = tf.concat([mfccs_0, log_frame_energy, mfccs_delta_1, energy_delta_1, mfccs_delta_2, energy_delta_2], axis=-1)

    return mfccs

Now let's plot the power spectrogram (in log-scale) of the same noisy audio shown above and its MFCCs coefficients.

The left plot shows around which frequencies of the spectrum the energy (color bar) is more concentrated (the perception of loudness by the human auditory system scales logarithmically with the power of the sound wave). The y axis represents the range of frequencies of the recording; each bin is a frequency level, from 0 up to the Nyquist frequency.

The right plot shows which of the MFCCs coefficient are more relevant. The first coefficient (not present in the plot) represents loudness and is often discarded because not informative in speech recognition.

In [ ]:
spec, wav = get_spectrogram(noisy_audio_sample, SAMPLE_RATE)
mfccs = get_mfccs(*apply_mel_filterbanks(spec, wav, SAMPLE_RATE))

plot_spectrogram_and_mfccs(spec, mfccs)

In [ ]:
print('Spectrogram shape: ', str(spec.shape).strip('()'), end = '\n')
print('MFCCs shape: ', str(mfccs.shape).strip('()'))

In [ ]:
#We define a varaiable we will use later taking the number of frames from the mffcs
N_FRAMES = mfccs.shape[0]

### Notes on SpecAugment

Often, the spectrograms are further pre-processed with a technique called *SpecAugment*. SpecAugment consists of three specific deformations applied directly to the log-mel spectrogram:

1. **Time Warping**

    It randomly stretches or squeezes parts of the image along the time axis, simulating speaking faster or slower at different points in the sentence.

2. **Frequency Masking**

    A mask (i.e., a block of zero values) is applied to a range of consecutive frequencies.
    It forces the model to identify words even if specific frequency information is missing.

3. **Time Masking**

    A mask is applied to a range of consecutive time steps.
    It forces the model to rely on context to recognize a specific word, even if part of the word itself is not present

In this lab, we won’t use this technique, but it’s useful to know in case you work with audio data in the future.

**Link to the original paper**: [SpecAugment: A Simple Data Augmentation Method for Automatic Speech Recognition](https://arxiv.org/abs/1904.08779).

### 1.7 - Adjacency matrix creation

Since our model is based on Graph Neural Networks (GNNs), we need to represent the temporal structure of the audio signal as a graph. In our case, each graph corresponds to one spoken word in the dataset and the MFCC coefficients represent the nodes of the graph. The connectivity structure of these nodes is encoded in the adjacency matrix, which specifies which frames can exchange information during message passing.

The base adjacency matrix is built by connecting MFCC frames that are close in time. Specifically, two nodes $i$ and $j$ are connected if the temporal distance $|i-j|$ is within a chosen `window_size`. This produces a local graph that captures **local temporal dependencies** in the speech signal.

However, local edges alone **limit the receptive field** of the GNN. To allow the model to aggregate information over longer temporal distances, we also compute **dilated** adjacency matrices. A dilated graph keeps only edges corresponding to nodes that are exactly $d$ hops apart in the original graph. This is implemented in `create_dilated_adjacency_matrix()`, where we:

 - Compute powers of the adjacency matrix $A, A^2, \dots, A^d$, where $A^k$ encodes all paths of length $k$.
 - Keep only the entries in $A^d$ that **do not** appear in any lower-order power.
 - Remove self-loops and binarize the matrix.

The function `create_adjacency_matrix()` then stacks the base adjacency matrix plus all dilated versions, providing the GNN with multiple resolutions of temporal connectivity.

In [ ]:
def create_dilated_adjacency_matrix(adjacency_matrix, dilation_rate):

    powers = [adjacency_matrix] # where to store all matrix prowers, starting from powers[0] = A^1

    A_power = adjacency_matrix

    for k in range(1, dilation_rate):
        # START CODE HERE (1 line) - compute A^(k+1)  using tf.linalg.matmul()
        A_power = None
        # END CODE HERE
        powers.append(A_power)

    A_d = powers[-1]

    # remove all lower-hop connections: A^1, A^2, ..., A^(d-1)
    for p in powers[:-1]:
        A_d = tf.where(p > 0, tf.zeros_like(A_d), A_d)

    # Remove self-loops from the matrix setting the diagonal to zeros
    # START CODE HERE (1 line) - set the diagonal of A_d to zero using tf.linalg.set_diag()
    A_d = None
    # END CODE HERE

    # binarize the adj matrix (keep all entries > 0)
    # START CODE HERE (1 line) - Convert boolean positivity condition to float using tf.cast()
    A_d = tf.cast(A_d > 0, tf.float32)
    # END CODE HERE

    return A_d


def create_adjacency_matrix(mfcc, num_frames, label, n_dilation_layers=0, window_size=5):

    adjacency_matrices = []
    indices = tf.range(num_frames, dtype=tf.int32)
    i = tf.reshape(indices, [-1, 1])
    j = tf.reshape(indices, [1, -1])

    # START CODE HERE (1 line) - compute |i - j| using tf.abs()
    distance = None
    # END CODE HERE

    # create adjacency by thresholding with window_size
    adjacency_matrix = tf.cast(distance <= window_size, tf.float32)

    # Remove self-loops
    adjacency_matrix = tf.linalg.set_diag(
            adjacency_matrix,
            tf.zeros(num_frames, dtype=tf.float32)
        )

    adjacency_matrices.append(adjacency_matrix)

    dilation_rate = 2

    for layer in range(n_dilation_layers):

        # START CODE HERE (1 line) - call the helper function create_dilated_adjacency_matrix
        dilated_A = None
        # END CODE HERE

        adjacency_matrices.append(dilated_A)

        dilation_rate += 2  # Increase for next layer

    return adjacency_matrices

In [ ]:
# computing adjacency matrices for the mffcs from our audio sample.
example_adjacency_matrix = create_adjacency_matrix(mfccs, 98, 0, 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# Adjacency matrix
axes[0].imshow(example_adjacency_matrix[0], cmap='viridis', interpolation='nearest')
axes[0].set_title("Adjacency Matrix (Window-based)")
axes[0].set_xlabel("Node index")
axes[0].set_ylabel("Node index")
fig.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046, pad=0.04)

# Dilated adjacency matrix ---
axes[1].imshow(example_adjacency_matrix[1], cmap='viridis', interpolation='nearest')
axes[1].set_title("Dilated Adjacency Matrix")
axes[1].set_xlabel("Node index")
axes[1].set_ylabel("Node index")
fig.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

### 1.8 Conversion of MFCCs to graph tensor

Next, you will implement the `mfccs_to_graph_tensors_for_dataset()` function.
- First, we reduce the node representation, if needed.
- Then, we create the node set creating the following dictionary using the `tfgnn.NodeSet.from_fields()` function:

```
{
"frames": tfgnn.NodeSet.from_fields(features={"features": mfcc_static},
sizes=[tf.shape(mfcc_static)[0]])
  }
```
- We will create the edge set with unique names using the `tfgnn.EdgeSet.from_fields()`

```
edge_sets[edge_set_name] = tfgnn.EdgeSet.from_fields(
            features={"weights" : weights},
            sizes=[tf.shape(edges)[0]],
            adjacency=tfgnn.Adjacency.from_indices(
                source=("frames", sources),
                target=("frames", targets)
            )
        )
```
- Finally, we will create the graph tensor using the `tfgnn.GraphTensor.from_pieces()`, by specifying inside the node set and the edge set.




In [ ]:
def mfccs_to_graph_tensors_for_dataset(mfcc, adjacency_matrices, label, reduced_node_bool, reduced_node_k):

    # reduce the node representation
    if reduced_node_bool:
        if ((98 // reduced_node_k) == (98/ reduced_node_k)):
          mfcc_static = tf.reshape(mfcc, [98 // reduced_node_k, 39])
        else:
          mfcc_static = tf.reshape(mfcc, [((98 // reduced_node_k) + 1), 39])
    else:
        mfcc_static = tf.reshape(mfcc, [98, 39])

    # Create the node set
    # START CODE HERE (1 line)
    node_sets = None
    ### END CODE HERE

    # Create an edge set for each adjacency matrix
    edge_sets = {}

    # Unstack the matrices so we can iterate over them
    unstacked_matrices = tf.unstack(adjacency_matrices, axis=0)

    for i, adjacency_matrix in enumerate(unstacked_matrices):
        # Get edges from this adjacency matrix
        edges = tf.where(adjacency_matrix > 0)

        # Get corresponding weights
        weights = tf.gather_nd(adjacency_matrix, edges)

        # Extract source and target indices
        sources = edges[:, 0]
        targets = edges[:, 1]

        # Create edge set with unique names
        edge_set_name = f"connections_{i}"
        # START CODE HERE (1 line)
        edge_sets[edge_set_name] = None
        ### END CODE HERE

    # Create the graph tensor with all node sets and edge sets
    # START CODE HERE (1 line)
    graph_tensor = None
    ### END CODE HERE

    return graph_tensor

In [ ]:
# create useful global variables
BATCH_SIZE = 64

# Set the number of dilation layers (i.e. k creates the undilated adjacency matrix and k-1 dilated adjacency matrices)
N_DILATION_LAYERS = 5

# Whether to reduce the node representation of the graph (i.e. pooling over the 98 frames in groups of size k)
REDUCED_NODE_REP_BOOL = False
REDUCED_NODE_REP_K = 0

# Parameters for the adjacency matrix creation
# Simple sliding window size (for 'window')
WINDOW_SIZE_SIMPLE = 5

## 2 - Data Loading using the Dataset API

At this point, you can easily apply the previously defined methods to the entire dataset and the results will maybe still fit in your (RAM) memory.

But what if this is not the case?

Let's suppose that the dataset does not entirely fit in your memory, and maybe your preprocessing pipeline is computationally intensive.

Under these circumstances, you have to read data from files (maybe stored in a remote server) every time you need a new sample, which can easily be the main bottleneck of the entire training process. A proper and optimized data-loading pipeline is of primary importance in many cases. As you have seen in Lab 2, you can exploit the Tensorflow Dataset API, which exposes many useful functions to assist you in this process (documentation [here](https://www.tensorflow.org/guide/datasets)).

The main goal of this phase is to create a `tf.data.Dataset` object to efficiently load and preprocess your data.

First, let's put together the loading and preprocessing methods in a single function `create_tf_dataset`.

**NOTE**: As you have seen in Labs 2, TensorFlow takes as input 32-bit floating point data usually; when using index encoding instead of one-hot encoding instead, you should pass the label vector as 32-bit integer. Make sure to return the proper data type to be fed to the network. The use of 64-bit preicison would make the network training extremely inefficient on most GPUs. Furthermore, that type of precision is not typically required for network training.

#### Notes on Data Augmentation

The order in which you use the ```Dataset``` API methods is relevant and might vary depending on the context. In particular, when training complex neural networks, often you want to perform some kind of ***data augmentation***, and the API methods can be usefully applied. Data augmentation consists of applying some transformations (such as the ones shown in this notebook) to the input data and present the model with the same data point slightly perturbed, as might occur in a real context, in order to improve the generalization and make it invariant to slight changes of the input. However, simply implementing a pipeline such that:
$\boldsymbol{f: X \rightarrow X^{'}, \quad X, X^{'} \subset \mathbb{R}^{n}, \quad |X| \ll |X^{'}|}$, where $\boldsymbol{X, X^{'}}$ are respectively the set of original and augmented input data, can increase the number of training samples too much. Instead, what is usually done is presenting the model with the same number of samples (or just a bit more) at each epoch, but transformations (including identity) are randomly applied to each sample on the go. This strategy effectively presents the model with an increased number of data without taxing too much storage and computational resources (to be correct, here there is a trade-off with CPU load, that is effectively the component that applies the transformations).

### 2.1 - Dataset definition
Now, let's define the `create_dataset` function following the same steps in Lab 2.

#### - **Create a Dataset object**
We use the method `from_tensor_slices` to instantiate a `tf.data.Dataset` object, which iteratively gets data from given tensors (documentation [here](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices)).
While in Lab 2 we implemented an unsupervised learning technique (autoencoder) in this lab we want to implement a classifier (supervised learning) and, in turn, we need to include labels in our Dataset.
Specifically, we use as arguments a tuple composed of the `file_names` list (\['string_1', 'string_2']) and the `labels` list (\[34, 2, 0,...\]), both extracted from the `df` dataframe.

#### - **Map the read_path_to_wav function**
In this case, we want to transform the filename string to a 32-bit tensor containing the actual preprocessed data using the `read_path_to_wav` function.
We use the `map()` method (documentation [here](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#map)), that usually is combined with `tf.numpy_function` (documentation [here](https://www.tensorflow.org/api_docs/python/tf/numpy_function)) to include arbitrary python functions inside the TensorFlow graph. However, in this case we are working with ```tfgnn.GraphTensor``` objects, this create issues with tf.numpy_function and as you can see the code pipeline include tf methods to avoid the need to use `tf.numpy_functions`.

The `map()` function converts your dataset entries from (filename, label) to (preprocessed ECG data, label).

#### -  **Map all other pre-processing functions**

#### - **Cache the dataset**
In this example, both the baseline wander removal and the normalization steps can be performed a single time on the entire dataset, since the result is always the same. Thus, we use caching to store partial results both in a file or in memory (RAM), documentation [here](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#cache).

**IMPORTANT NOTE**: Remember that the order of the dataset transformations is very important, especially when using a caching system. For example, if you applied the cache **after** the random noise injection, all your sample would be perturbed with noise only the first time, and stored in the cache. All the subsequent calls would have always returned the same exact crop, resulting in both a waste of data and more overfitting issues.

#### - **Shuffle**
Documentation [here](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle).

#### - **Repeat the dataset indefinitely**
Documentation [here](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#repeat).

#### - **Batch**
Put together *batch_size* samples into a single batch of data, which will be the input of the network. The shape of the data changes from (n_samples, 1) to (batch_size, n_samples, 1).

#### - **Prefetch**
With this operation we go from this situation:

<img src="https://drive.google.com/uc?export=view&id=1MTOXGXgTrYWF0nxLReGoOjYKLHz99FCW" width="600"/>

to:

<img src="https://drive.google.com/uc?export=view&id=1zWErI7Y2T-G6Cd3uNZyDcvCwUTpUNd-B" width="600"/>

We will skip the caching of the training set as it would require a lot of time, but you can notice that the caching operation should be performed **before** applying the random data augmentation. However, this will increase a lot training time; we can also cache the training set in the same way we do for the validation set by changing the code, but the transformation performed will be fixed in this case.

In [ ]:
def create_tf_dataset(dataframe, sample_rate, background_noise_files, noise_prob, cache_file = '', batch_size = 16, cache = False, shuffle = True, repeat = False, noise = False):

  # obtain file names and numerical labels
  # START CODE HERE (1 line)
  file_names, labels = None
  ### END CODE HERE

  # Create the Dataset object
  # START CODE HERE (1 line)
  dataset = None
  ### END CODE HERE

  if shuffle:
    # START CODE HERE (1 line)
    dataset = None
    ### END CODE HERE

  # Loading and decoding wav files using read_path_to_wav() and then map the function to the dataset
  # START CODE HERE (2 lines)
  read_path_to_wav_lambda = None
  dataset = None
  ### END CODE HERE

  # Adjust audio length using adjust_audio_length() and then map the function to the dataset
  # START CODE HERE (2 lines)
  adjust_audio_length_lambda = None
  dataset = None
  ### END CODE HERE

  # Caching the dataset after deterministic transformation (Optional)
  # if cache:
  #   dataset = dataset.cache(cache_file).shuffle(1000)

  if noise:
    #Applying time-shift using apply_time_shift() and then map the function to the dataset
    # START CODE HERE (2 lines)
    apply_time_shift_lambda = None
    dataset = None
    ### END CODE HERE

    #Applying background noise using apply_random_noise() and then map the function to the dataset
    # START CODE HERE (2 lines)
    apply_random_noise_lambda = None
    dataset = None
    ### END CODE HERE

  # Compute the spectrogram using get_spectrogram
  # use *get_spectrogram(wav, sample_rate) as the function returns a tuple of elements - more compact version of get_spectrogram(wav, sample_rate)[0], get_spectrogram(wav, sample_rate)[1]
  # START CODE HERE (2 lines)
  get_spectrogram_lambda = None
  dataset = None
  ### END CODE HERE

  # Apply the Mel filters
  # use *apply_mel_filterbanks(spec, wav, sample_rate) - same as above
  # START CODE HERE (2 lines)
  apply_mel_filterbanks_lambda = None
  dataset = None
  ### END CODE HERE

  # Compute MFCC + Delta features using the get_mfccs() function
  # START CODE HERE (2 lines)
  get_mfccs_lambda = None
  dataset = None
  ### END CODE HERE

  # Create adjacency matrix using the create_adjacency_matrix() function
  # START CODE HERE (2 lines)
  create_adjacency_matrix_lambda = None
  dataset = None
  ### END CODE HERE

  # Convert MFCCs to graph tensor using mfccs_to_graph_tensors_for_dataset() function
  # START CODE HERE (2 lines)
  mfccs_to_graph_tensors_for_dataset_lambda = None
  dataset = None
  ### END CODE HERE

  if cache:
      dataset = dataset.cache(cache_file)

  if repeat or is_train:
    dataset = dataset.repeat()
    print('Repeated dataset')

  dataset = dataset.batch(batch_size = batch_size)
  print('Batched dataset')

  dataset = dataset.prefetch(buffer_size = 1)
  print('Prefetched dataset', end = '\n-----\n')

  return dataset

### 2.2 - Dataset initialization
Now we can finally define our training, validation and test dataset.

We also evaluate the number of steps required to load and process the training set (num. of  training samples / batch_size).

- Remember that the shuffle and noise flags should be *True* for the training set and *False* for the remaining sets, so to perform shuffle and data augmentation only for the first one.
- Moreover, cache **only the validation** set.

In [ ]:
# START CODE HERE (3 lines)
train_ds = None
valid_ds = None
test_ds = None
### END CODE HERE

 To save some time (the full creation of the caches may require about 30 minutes depending on your hardware), the caches of the train and validation sets are provided. Keep in mind that this is not a good practice and may cause some issues. Typically cache files should not be transferred among different machines.

 If you have problems with the provided cache files, just delete/rename them (all the files starting with *train_cache* and *val_cache*) and recreate your own caches.

In [ ]:
example_dataset = create_tf_dataset(val_df[:BATCH_SIZE*3], sample_rate= SAMPLE_RATE,
                             background_noise_files=background_noise_files,
                             noise_prob=0.8, cache_file = 'example_samples', batch_size = BATCH_SIZE, cache = True, repeat = True)
iterat = iter(example_dataset)
# Iterate for 3 epochs
for num_epoch in range(3):
    # Time the loading time
    it = time.time()
    for step in tqdm(range(3)):
        # Get the next batch of data
        next(iterat)
    # Print loading time
    print('EPOCH {} - Time to load the entire dataset [seconds]: {}'.format(num_epoch+1, time.time() - it))

 ## 3 - Model definition

In the next few cells we are going to define a simple architecture used to carry out KWS (Keyword Spotting) at a graph-level.

You are free to define your network architecture.

As a starting point, we propose the following baseline GNN that operates on MFCC sequences represented as graphs. Each utterance (i.e., each spoken word) is converted into a graph where:

 - **nodes** correspond to MFCC frames, and
 - **edges** encode temporal proximity or longer-range relations via dilated adjacency matrices.

<img src="https://drive.google.com/uc?export=view&id=1qrruFRQUQ0a1ol4HBBK_0jLiCzzIZv0F" style="width:50%">
<caption><center> All the nodes inside the window $[i - W, i + W]$ are connected with node i.  </center></caption>

<img src="https://drive.google.com/uc?export=view&id=1Pa4isDmipWi_DLfzvLdaB34baXK7dxkX" style="width:50%">
<caption><center> Dilation $d = 2$ connectivity pattern. At the second hidden layer each node ios connected with nodes at distance 4, the conenctivity distance increase exponentially with layers. </center></caption>

This graph representation allows the model to capture both local and non-local temporal dependencies.

Below is the detailed explanation of each block of the baseline GNN.

#### Input layer: (N_frames, N_features) + adjacency matrices
`tf.keras.layers.Input(type_spec = graph_tensor_specification)`
type_spec tells Keras what kind of Tensor the input is — not just its shape, but its entire structure. We use it when the input is not a standard dense tensor. For Graph Neural Networks, the input is not a single tensor, but a collection of related tensors: node features, edge list or adjacenti matrix, graph size, possibly multiple adjacency matrices (dilated...)

#### Convert into graph tensor
`graph = tfgnn.keras.layers.MapFeatures()(input_graph)`

tf-gnn expects all graph features to be in a canonical format.
MapFeatures() prepares the Graph Tensor to ensure this.

#### Check if the graph is batched and merge the components
TF-GNN models operate on a specific internal structure: a scalar GraphTensor, which conceptually represents a single graph. When a batch of graphs is passed to the model, TF-GNN requires that they are first merged into one unified GraphTensor where each original graph becomes a separate component inside this structure. Importantly, components are fully disconnected—no edges link them—so no information flows between graphs belonging to different batch elements. After merging, all nodes from all batch elements are stacked together. For example, a batch of 32 graphs with 98 nodes each becomes a single GraphTensor with 32×98=3136 nodes.

### 3.1 - Model architecture

#### Input vector

The input vector is passed through the network in two different ways:

- A **'normal'** one, where the mfccs are just concatenated and then go through a dense layer, or
- A **'splitted'** where the static, delta and delta-delta mfccs are separated in three different vectors and each vector is passed through a different dense layer, then the three outputs are concatenated together and go through another dense layer.


#### Message passing and aggregation function
At each layer, the information arriving to each node from its neighbors needs to be compressed into a single representation used as the node state for the next layer. With this mechanism, every node get information from progressively more distant neighbors as the network goes deeper (this happens even without any dilation given enough layers). To do this, an aggregation function should be defined; many alternatives are available, in this code we use the average of the state nodes. The resulting representation is then passed through a dense layer, and the output is concatenated with the target node state vector and then passes again through a dense layer. Each dense layer uses dropout, layer normalization and ReLu.
  
Depending on the number of dilation levels, the model may include more graph convolution layers, each working on a different adjacency matrix. By default the ```create_adjacency_matrix()``` use a dilation 2 step(e.g., dilation 2, dilation 4, etc.), this means that each message passing layer increases is dilation by two over the previous.

<img src="https://drive.google.com/uc?export=view&id=1tGMzCTbzeveb8C2kjFpgCC3BpB2gTofQ" style="width:50%">
<caption><center> Message passing pipeline </center></caption>

#### Pooling Layer (Global or Attention-based)

After the graph convolutions, we must define a readout layer to convert the sequence of node embeddings into a single vector representing the entire word.
Common choices include:

- Global Average Pooling over nodes
- Global Max Pooling
- Attention-based pooling

in the functtion `base_model()` is controlled by the parameter `context_mode`.

#### Dense output layer
Finally, a fully connected Dense layer maps the pooled graph representation to a logit vector for multi-class classification.


Layers' documentation [here](https://github.com/tensorflow/gnn).

In [ ]:
def base_gnn(graph_tensor_specification, initial_nodes_mfccs_layer_dims = 64,
            initial_edges_weights_layer_dims = [16], message_dim = 128,
            next_state_dim = 128, num_classes = 35, l2_reg_factor = 6e-6,
            dropout_rate = 0.2, use_layer_normalization = True,
            n_message_passing_layers = 4, dilation = False, n_dilation_layers = 2,
            context_mode = 'mean', initial_state_mfcc_mode = 'normal'):


    # Input layer
    # START CODE HERE ### (1 line)
    input_graph = None
    ### END CODE HERE ###

    # Encode input features
    # START CODE HERE ### (1 line)
    graph = None
    ### END CODE HERE ###

    # Check if the input GraphTensor is batched
    is_batched = (graph.spec.rank == 1)
    if is_batched:
        graph = graph.merge_batch_to_components()

    # Helper function
    # Dense layer + Dropout + optional layer regularizer
    def dense(units, use_layer_normalization = False):
            regularizer = tf.keras.regularizers.l2(l2_reg_factor)
            result = tf.keras.Sequential([
                tf.keras.layers.Dense(
                    units,
                    activation = "relu",
                    use_bias = True,
                    kernel_regularizer = regularizer,
                    bias_regularizer = regularizer),
                tf.keras.layers.Dropout(dropout_rate)])
            if use_layer_normalization:
                result.add(tf.keras.layers.LayerNormalization())
            return result

    # Initialize hidden states for nodes
    def set_initial_node_state(node_set, node_set_name):

        # helper function
        def dense_inner(units, use_layer_normalization = False, normalization_type = "normal"):
            regularizer = tf.keras.regularizers.l2(l2_reg_factor)
            result = tf.keras.Sequential([
                tf.keras.layers.Dense(
                    units,
                    activation = "relu",
                    use_bias = True,
                    kernel_regularizer = regularizer,
                    bias_regularizer = regularizer),
                tf.keras.layers.Dropout(dropout_rate)])
            if use_layer_normalization:
                if normalization_type == 'normal':
                    result.add(tf.keras.layers.LayerNormalization())
                elif normalization_type == 'group':
                    result.add(tf.keras.layers.GroupNormalization(message_dim))
            return result

        if initial_state_mfcc_mode == 'normal':
            return tf.keras.layers.Dense(initial_nodes_mfccs_layer_dims, activation="relu")(
                    node_set["features"]
                )

        else:
                # Split the diff. features such that we can do separate layer learning

                features = node_set["features"]

                base_mfccs = features[: , 0:12]
                delta_mfccs = features[: , 13:25]
                delta_delta_mfccs = features[:, 26:38]
                energy_features = tf.concat([features[:, 12:13], features[:, 25:26], features[:, -1:]], axis=-1)

                base_processed = dense_inner(24, use_layer_normalization=True)(base_mfccs)
                delta_processed = dense_inner(24, use_layer_normalization=True)(delta_mfccs)
                delta_delta_processed = dense_inner(24, use_layer_normalization=True)(delta_delta_mfccs)
                energy_processed = dense_inner(8, use_layer_normalization=True)(energy_features)

                # Concatenate the processed features
                combined_features = tf.keras.layers.Concatenate()(
                    [base_processed, delta_processed, delta_delta_processed, energy_processed]
                )

                return dense_inner(initial_nodes_mfccs_layer_dims, use_layer_normalization=True)(combined_features)

    graph = tfgnn.keras.layers.MapFeatures(
        node_sets_fn = set_initial_node_state, name = 'init_states')(graph)

    def convolution(message_dim, receiver_tag):

        # Set receiver feature to None such that we don't concatenate the receiver node's features (we do that in nextstatefromconcat)
        return tfgnn.keras.layers.SimpleConv(dense(message_dim), "sum", receiver_tag = receiver_tag, receiver_feature= None)

    def next_state(next_state_dim, use_layer_normalization):

        # return next state using tfgnn.keras.layers.NextStateFromConcat() with argument dense(next_state_dim, use_layer_normalization=use_layer_normalization)
        # START CODE HERE (1 line)
        next_s = None
        ### END CODE HERE
        return next_s

    if not dilation:
        # Like this, in the modulo calculation, we only use connections_0 all the time, i.e. we do not use dilation
        n_dilation_layers = 1

    for i in range(n_message_passing_layers):
        dil_layer_num = i % n_dilation_layers # circular usage of dilated adjacency matrices throughout message passing layers
        graph = tfgnn.keras.layers.GraphUpdate(
            node_sets = {
                "frames" : tfgnn.keras.layers.NodeSetUpdate(
                    {f"connections_{dil_layer_num}" : convolution(message_dim, tfgnn.SOURCE)},
                next_state(next_state_dim, use_layer_normalization)
                )
            }
        )(graph)


    # Take all the 98 learnt node features , aggregate them using sum
    # which is then representing the context vector (i.e. the "graph node")
    pooled_features = tfgnn.keras.layers.Pool(
        tfgnn.CONTEXT, context_mode, node_set_name = "frames")(graph)
    logits = tf.keras.layers.Dense(num_classes)(pooled_features)

    model = tf.keras.Model(input_graph, logits)

    return model

In [ ]:
# Check the shape of the dataset
for graph, label in train_ds.take(1):
    print(f"Graph shape: {graph.shape}")
    print(f"Label shape: {label.shape}")
    print(label)

    # We need to get the graphs_spec for our model input
    graphs_spec = graph.spec

base_model = base_gnn(graph_tensor_specification = graphs_spec,
                            n_message_passing_layers = 5,
                            initial_nodes_mfccs_layer_dims= 32,
                            next_state_dim = 32,
                            message_dim = 32,
                            initial_state_mfcc_mode = 'splitted',
                            context_mode = 'mean',
                            dropout_rate = 0.2,
                            dilation = False,
                            n_dilation_layers = 0,
                            l2_reg_factor= 1e-4,
                            )

In [ ]:
base_model.summary()

 ## 4 - Model training and testing
 ### 4.1 - Model training
 Now let's train the model for some epochs. Then plot the loss values and the accuracy during training.



Here we use the early stopping, learning rate reduction and model checkpoints [callbacks]('https://www.tensorflow.org/api_docs/python/tf/keras/callbacks').

In [ ]:
def train(model, train_ds, val_ds, steps_per_epoch, epochs = 50, batch_size = 32, use_callbacks = True, learning_rate = 0.001):

    # Define callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=2,
            min_lr=1e-10,
            verbose=1
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath='best_model_weights.h5',
            monitor='val_loss',
            save_best_only=True,
            save_weights_only=True,
            verbose=1)
    ]


    model.compile(
          optimizer = tf.keras.optimizers.legacy.Adam(learning_rate = learning_rate),
          # using Sparse categorical crossentropy
          loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),
          metrics = [tf.keras.metrics.SparseCategoricalAccuracy()]
      )


    if use_callbacks:
        history = model.fit(train_ds, validation_data = val_ds, epochs = epochs, steps_per_epoch = steps_per_epoch, callbacks = callbacks)
    else:
        history = model.fit(train_ds, validation_data = val_ds, epochs = epochs, steps_per_epoch = steps_per_epoch)

    return history


Given that the training is very long due to the heavy data pipeline introduced, we suggest you to load the model uncommenting the code in the cell below and check the evaluation metrics directly.

In [ ]:
# import json

# base_model.load_weights('best_model_weights_2.h5')

# with open('history.json', 'r') as f:
#     history_dict = json.load(f)

In [ ]:
num_epochs = 30

SAMPLE_PER_EPOCH = 19000

# Calculate steps per epoch
train_steps = SAMPLE_PER_EPOCH // BATCH_SIZE

history = train(base_model, train_ds, valid_ds, steps_per_epoch = train_steps, epochs = num_epochs, batch_size = BATCH_SIZE, use_callbacks = True)

You can see from the plot that the validation performance is higher than the training performance. This is normal during the first few epochs, especially when the model uses dropout or other regularization techniques (assuming the dataset is properly split). This behaviour arises from several factors:

1. TensorFlow reports training metrics as running averages over the batches within an epoch, meaning the displayed training values reflect predictions made at multiple intermediate weight states. In contrast, validation metrics are computed only once at the end of the epoch using the final model weights.

2. Data augmentation and regularization are applied only during training. Validation data are evaluated without augmentation, and regularization layers are disabled, resulting in an effectively higher model capacity during validation.

3. Persistent divergence between training and validation curves may indicate over-regularization. If the gap doesn’t shrink over time, you may need to reduce dropout, lower L2 regularization, or adjust the pre-processing (e.g., reduce noise injection).

Normally, even with these factors, the training and validation curves eventually tend to converge. Since this is not happening here, it suggests that the model may be regularized too heavily.

In [ ]:
# Plot loss
plt.figure()
plt.plot(history_dict['loss'], label='Train loss')
plt.plot(history_dict['val_loss'], label='Val loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')

# Plot accuracy
plt.figure()
plt.plot(history_dict['sparse_categorical_accuracy'], label='Train accuracy')
plt.plot(history_dict['val_sparse_categorical_accuracy'], label='Val accuracy')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

If you want to test the hypotheses made in the cells above, here you can define a model without regularization or a training set without noise (the training will be much faster than before).

In [ ]:
# train_ds = create_tf_dataset(train_df, sample_rate = SAMPLE_RATE,
#                              background_noise_files=background_noise_files,
#                              noise_prob=0.8, cache_file = 'training_set', batch_size = BATCH_SIZE, cache = True)

# base_model_more_complex = base_gnn(graph_tensor_specification = graphs_spec,
#                             n_message_passing_layers = 5,
#                             initial_nodes_mfccs_layer_dims= 32,
#                             next_state_dim = 32,
#                             message_dim = 32,
#                             initial_state_mfcc_mode = 'splitted',
#                             context_mode = 'mean',
#                             dropout_rate = 0,
#                             dilation = False,
#                             n_dilation_layers = 0,
#                             l2_reg_factor= 0
#                             )

# history = train(base_model_more_complex, train_ds, valid_ds, steps_per_epoch = train_steps, epochs = num_epochs, batch_size = BATCH_SIZE, use_callbacks = True)

## 5 - Some validation metrics

Using only accuracy as a metric in a situation with multiple classes not necessarily equally represented might hide some deficiencies of the network in its ability to learn specific parts of the label space. Given that the traditional metrics used in binary classification (like the ones used in Laboratory 3) cannot be directly applied in a multiclass context we shall resort to a version of such metrics properly adapted to the context.

Here we have different alternatives:

- We can compute the usual metrics (precision, recall, f-1 score) for each class; this gives a more detailed image of what the gaps in our model predictions are, but can be harder to read and in general not feasible in the presence of many classes.

- We can compute some kind of aggregation of the mentioned metrics (possibly with different weighting of each class), resulting in a single value that describes the behavior of the network at prediction.

*Scikit-learn* provides functions to carry out both of the strategies described
(the functions are the same for binary and multiclass classification, you only need to change some parameters)
(documentation [here](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics)).

Recap from Lab 4: first, we need the output of the network for all the test samples. To obtain it use:
```python
test_preds = model.predict(test_dataset)
```
then, we have to get which of the classes is the one with the highest probability according to the classifier, that will be the model's prediction:
```python
tf.argmax(test_preds, axis = 1)
```

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_curve, auc

# Get the test labels
test_labels = test_df['label'].values

# Get the network output for the test set
### START CODE HERE ### (1 line)
test_preds = None
### END CODE HERE ###

# Get the estimated classes
### START CODE HERE ### (1 line)
test_est_classes = None
### END CODE HERE ###

In [ ]:
# Evaluate accuracy
### START CODE HERE ### (1 line)
accuracy = accuracy_score(test_labels, test_est_classes)
### END CODE HERE ###

# Use average='weighted' to account for class imbalance
precision, recall, fscore, _ = precision_recall_fscore_support(
    test_labels,
    test_est_classes,
    average = 'macro'  # Use 'macro' if you want to treat all classes equally regardless of size
)

print('#### TEST PERFORMANCE')
print(f'Accuracy: {accuracy*100:.2f}%')
print(f'Precision: {precision*100:.2f}%')
print(f'Recall: {recall*100:.2f}%')
print(f'F-score: {fscore*100:.2f}%')

Finally, given that the number of classes is not too high, we can also plot the most common predictions for each class; in this way, we can check exactly for every class what's our model's predictions are and the most common prediction errors (similar to the first of the two strategies described above). Note that this is exactly the **Recall** measure class wise.

In [ ]:
plot_prediction_distribution(test_labels, test_est_classes, classes)

Congratulations! Lab 4 successfully completed :)